In [ ]:
import mcstasscript as ms
import mcstastox as mx
import scipp as sc
from scipp.typing import VariableLike
import scipp as sc
import matplotlib.pyplot as plt
from scippneutron.conversion.graph.beamline import beamline


import sys 
from trex_reduction import inelastic
from trex_reduction import produce_trex_event_object

import os

In [ ]:
cwd = os.getcwd()
file_path = cwd + "/isis_vandadium_180"

with mx.Read(file_path) as loaded_data:
    scipp_object = loaded_data.export_scipp_simple(source_name="SourceMantid",
                                            sample_name="iso_samp")
    
data = ms.load_data(file_path)

In [ ]:
with mx.Read(file_path) as loaded_data:
    print("=== All components ===")
    print(loaded_data.get_components())
    print("=== Components with data ===")
    print(loaded_data.get_components_with_data())
    print("=== Components with IDs ===")
    print(loaded_data.get_components_with_ids()) 
    print("=== Components with geometry ===")
    print(loaded_data.get_components_with_geometry())

In [ ]:
with mx.Read(file_path) as loaded_data:
    scipp_object = loaded_data.export_scipp_simple(source_name="SourceMantid",
                                            sample_name="iso_samp")
    
# Load event data into scipp 
event_object = scipp_object
# McStas provides absolute time, not time of flight
event_object.coords["tof"] = event_object.coords["t"]
# Add additional information required for inelastic scattering
event_object = produce_trex_event_object(event_object, file_path, "Monitor6")
event_object

In [ ]:
qens_graph = {**beamline(scatter=True), **inelastic}
sc.show_graph(qens_graph)

In [ ]:
event_object = event_object.transform_coords("dE", graph=qens_graph)
event_object = event_object.transform_coords("mag_Q", graph=qens_graph)
event_object = event_object.transform_coords("two_theta", graph=qens_graph)

event_object

In [ ]:
import numpy as np

In [ ]:
fig, ax = plt.subplots()
(event_object.hist(two_theta=51)).plot(ax=ax, linestyle='-', color='tab:blue')
ax.axvline(np.pi/2, color = 'k')
# (event_object.hist(theta_test=51)).plot(ax=ax, linestyle='--', color='tab:orange')
plt.show()

In [ ]:
(event_object.hist(mag_Q = 51)).plot()

In [ ]:
(event_object.hist(dE = 500)).plot()



In [ ]:
event_object.hist(two_theta = 20, dE = 200).plot()

In [ ]:
event_object.hist(mag_Q = 20, dE = 200).plot()

In [ ]:
import plopp as pp
%matplotlib widget

In [ ]:
pp.slicer(event_object.hist(two_theta=10, dE=500), keep=['dE'], logc=True)

In [ ]:

pp.slicer(event_object.hist(mag_Q=10, dE=500), keep=['dE'], logc=True)
# fig.children[0].ax.set_xlim(-4, 4)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import sys
sys.path.append("./plet_data")
from plet_data import PletData

plet_data = PletData('./plet_data/vanadium_362.nxspe', 3.62, omega_lims = [-2,2])


fig, ax = plt.subplots()

hist = event_object.hist(dE=25)
dE = hist.coords['dE'].values  # bin edges, length 501
n = hist.values                 # counts, length 500
ax.stairs(n / np.max(n) , dE, label = 'sim')
ax.stairs(plet_data.data.nansum('q').values / np.max(plet_data.data.nansum('q').values), plet_data.data.coords['omega'].values, label = 'pLET-exp')
ax.set_xlim([-0.4,0.4])



ax.set_xlabel('dE (meV)')
ax.set_ylabel('Normalised Intensity')
ax.legend()
ax.set_title('Vanadium')
plt.show()

In [ ]:
fig,[ax1,ax2] = plt.subplots(1,2,figsize = (8,4))
fig.subplots_adjust(wspace=0.5)

plet_data.data.masks.clear()
plet_data.data.plot(ax=ax1)
ax1.set_xlim([-0.4,0.4])
ax1.set_ylim([0.5,2.5])
ax1.set_title('pLET')


event_object.hist(mag_Q = 40, dE = 25).plot(ax=ax2)
ax2.set_xlim([-0.4,0.4])
ax2.set_ylim([0.5,2.5])
ax2.set_title('McStas')
plt.show()

